In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import re

In [2]:
movies_df = pd.read_csv(r'C:\PycharmProjects\PythonProject\recommendation_for_films\dataset\TMDB_movie_dataset_v11.csv')
tv_df = pd.read_csv(r'C:\PycharmProjects\PythonProject\recommendation_for_films\dataset\archive\TMDB_tv_dataset_v3.csv')

In [3]:
tv_df = tv_df.rename(columns={'name': 'title'})

In [4]:
def parse_genres(genres_str):
    try:
        if pd.isna(genres_str) or genres_str is None or not genres_str.strip():
            return []
        if isinstance(genres_str, str):
            genres_list = [genre.strip() for genre in genres_str.split(',') if genre.strip()]
            return genres_list
        if isinstance(genres_str, list):
            return [g for g in genres_str if isinstance(g, str) and g]
        return []
    except Exception as e:
        return []

In [5]:
# Функция для расчета weighted_rate (5.0 для отсутствующих оценок)
def calculate_weighted_rate(row, min_vc, is_new_release=False):
    v_count = row.get('vote_count', 0)
    v_avg = row.get('vote_average', 0)
    
    # Для новых релизов используем более мягкую формулу
    if is_new_release:
        if v_count == 0 or pd.isna(v_count):
            # Для новых фильмов без оценок даем нейтральный рейтинг
            return 6.5  # Чуть выше среднего, чтобы они попали в выборку
        min_vc = min_vc / 4  # Снижаем требования для новых фильмов
    
    if v_count == 0 or pd.isna(v_count) or pd.isna(v_avg):
        return 5.0
    
    return (v_count / (v_count + min_vc)) * v_avg + (min_vc / (v_count + min_vc)) * 5.0

In [6]:
def create_combined(row):
    if isinstance(row['genres'], list):
        genres_str = ', '.join(row['genres']) if row['genres'] else 'Unknown'
    else:
        genres_str = str(row['genres']) if pd.notna(row['genres']) else 'Unknown'
    
    if 'release_date' in row and pd.notnull(row['release_date']):
        media_type = 'Movie'
        date_str = row['release_date'].strftime('%Y-%m-%d')
    elif 'first_air_date' in row and pd.notnull(row['first_air_date']):
        media_type = 'TV Series'
        date_str = row['first_air_date'].strftime('%Y-%m-%d')
    else:
        media_type = 'Unknown'
        date_str = 'No Date'
    
    return f"Title: {row['title']} ({media_type}, {date_str}). Overview: {row['overview']} Genres: {genres_str}. Rating: {row['weighted_rate']:.2f}"

In [7]:
def extract_year_from_combined(combined_str):
    try:
        match = re.search(r'\(.*?, (\d{4})-\d{2}-\d{2}\)', combined_str)
        return int(match.group(1)) if match else None
    except:
        return None

In [8]:
# Шаг 1: Предобработка фильмов (основной датасет >= 2020)
# # Обработка дат
movies_df['release_date'] = pd.to_datetime(movies_df['release_date'], errors='coerce')
tv_df['first_air_date'] = pd.to_datetime(tv_df['first_air_date'], errors='coerce')

In [9]:
# Разделение на старые и новые релизы
movies_df['year'] = movies_df['release_date'].dt.year
tv_df['year'] = tv_df['first_air_date'].dt.year

print(f"Исходный размер: Фильмы - {len(movies_df)}, Сериалы - {len(tv_df)}")

Исходный размер: Фильмы - 1288590, Сериалы - 168639


In [10]:
# ОБРАБОТКА СТАРЫХ ФИЛЬМОВ (2020-2023) - строгие фильтры
old_movies = movies_df[(movies_df['year'] >= 2020) & (movies_df['year'] <= 2023)].copy()
old_tv = tv_df[(tv_df['year'] >= 2020) & (tv_df['year'] <= 2023)].copy()

In [11]:
# Применяем строгие фильтры для старых фильмов
old_movies = old_movies[
    (old_movies['vote_count'] > 50) &  # Минимум 50 голосов для старых
    (old_movies['vote_average'] > 0) &
    (old_movies['title'].notna()) &
    (old_movies['overview'].notna()) &
    (old_movies['overview'].str.len() > 10) &
    (old_movies['genres'].notna())
]

old_tv = old_tv[
    (old_tv['vote_count'] > 50) &
    (old_tv['vote_average'] > 0) &
    (old_tv['title'].notna()) &
    (old_tv['overview'].notna()) &
    (old_tv['overview'].str.len() > 10) &
    (old_tv['genres'].notna())
]

print(f"Старые (2020-2023) после фильтров: Фильмы - {len(old_movies)}, Сериалы - {len(old_tv)}")

Старые (2020-2023) после фильтров: Фильмы - 2970, Сериалы - 1079


In [12]:
# ОБРАБОТКА НОВЫХ ФИЛЬМОВ (2024-2025) - мягкие фильтры
new_movies = movies_df[(movies_df['year'] >= 2024) & (movies_df['year'] <= 2025)].copy()
new_tv = tv_df[(tv_df['year'] >= 2024) & (tv_df['year'] <= 2025)].copy()

In [13]:
# Мягкие фильтры для новых фильмов
new_movies = new_movies[
    (new_movies['title'].notna()) &
    (new_movies['overview'].notna()) &
    (new_movies['overview'].str.len() > 10) &
    (~new_movies['overview'].str.contains('No Description|Plot unknown', case=False, na=False)) &
    (new_movies['genres'].notna())
]

new_tv = new_tv[
    (new_tv['title'].notna()) &
    (new_tv['overview'].notna()) &
    (new_tv['overview'].str.len() > 10) &
    (~new_tv['overview'].str.contains('No Description|Plot unknown', case=False, na=False)) &
    (new_tv['genres'].notna())
]

In [14]:
# Для новых фильмов без оценок устанавливаем дефолтные значения
new_movies['vote_count'] = new_movies['vote_count'].fillna(0)
new_movies['vote_average'] = new_movies['vote_average'].fillna(0)
new_tv['vote_count'] = new_tv['vote_count'].fillna(0)
new_tv['vote_average'] = new_tv['vote_average'].fillna(0)

print(f"Новые (2024-2025) после мягких фильтров: Фильмы - {len(new_movies)}, Сериалы - {len(new_tv)}")

Новые (2024-2025) после мягких фильтров: Фильмы - 28165, Сериалы - 260


In [15]:
# Парсинг жанров для всех
for df in [old_movies, old_tv, new_movies, new_tv]:
    df['genres'] = df['genres'].apply(parse_genres)
    # Фильтруем только те, где есть хотя бы один жанр
    df_len_before = len(df)
    df = df[df['genres'].apply(lambda x: len(x) > 0)]
    print(f"Удалено без жанров: {df_len_before - len(df)}")

Удалено без жанров: 0
Удалено без жанров: 0
Удалено без жанров: 0
Удалено без жанров: 0


In [16]:
# Пересоздаем датафреймы после фильтрации
old_movies = old_movies[old_movies['genres'].apply(lambda x: len(x) > 0)]
old_tv = old_tv[old_tv['genres'].apply(lambda x: len(x) > 0)]
new_movies = new_movies[new_movies['genres'].apply(lambda x: len(x) > 0)]
new_tv = new_tv[new_tv['genres'].apply(lambda x: len(x) > 0)]

In [17]:
# Удаление дубликатов
old_movies = old_movies.drop_duplicates(subset=['id'])
old_tv = old_tv.drop_duplicates(subset=['id'])
new_movies = new_movies.drop_duplicates(subset=['id'])
new_tv = new_tv.drop_duplicates(subset=['id'])

In [18]:
# Расчет weighted_rate с учетом типа релиза
min_vote_count = 130.0

old_movies['weighted_rate'] = old_movies.apply(
    lambda row: calculate_weighted_rate(row, min_vote_count, is_new_release=False), axis=1
)
old_tv['weighted_rate'] = old_tv.apply(
    lambda row: calculate_weighted_rate(row, min_vote_count, is_new_release=False), axis=1
)
new_movies['weighted_rate'] = new_movies.apply(
    lambda row: calculate_weighted_rate(row, min_vote_count, is_new_release=True), axis=1
)
new_tv['weighted_rate'] = new_tv.apply(
    lambda row: calculate_weighted_rate(row, min_vote_count, is_new_release=True), axis=1
)

In [19]:
# Создание combined для всех
for df in [old_movies, old_tv, new_movies, new_tv]:
    df['combined'] = df.apply(create_combined, axis=1)

In [20]:
# СТРАТЕГИЯ ОТБОРА
# Для старых фильмов - берем топ по weighted_rate с учетом popularity
old_movies_sorted = old_movies.sort_values(
    by=['weighted_rate', 'popularity'], 
    ascending=[False, False]
).head(3500)  # Берем больше старых популярных

old_tv_sorted = old_tv.sort_values(
    by=['weighted_rate', 'popularity'], 
    ascending=[False, False]
).head(3500)

In [21]:
# Для новых фильмов - берем ВСЕ или максимум с приоритетом по popularity
# Это гарантирует включение новых релизов
new_movies_sorted = new_movies.sort_values('popularity', ascending=False).head(1500)
new_tv_sorted = new_tv.sort_values('popularity', ascending=False).head(1500)

In [22]:
print(f"\nОтобрано для финального датасета:")
print(f"Старые фильмы (2020-2023): {len(old_movies_sorted)}")
print(f"Старые сериалы (2020-2023): {len(old_tv_sorted)}")
print(f"Новые фильмы (2024-2025): {len(new_movies_sorted)}")
print(f"Новые сериалы (2024-2025): {len(new_tv_sorted)}")


Отобрано для финального датасета:
Старые фильмы (2020-2023): 2970
Старые сериалы (2020-2023): 1079
Новые фильмы (2024-2025): 1500
Новые сериалы (2024-2025): 244


In [23]:
# Объединение всех данных
combined_df = pd.concat([
    old_movies_sorted[['title', 'genres', 'overview', 'weighted_rate', 'combined', 'popularity']],
    old_tv_sorted[['title', 'genres', 'overview', 'weighted_rate', 'combined', 'popularity']],
    new_movies_sorted[['title', 'genres', 'overview', 'weighted_rate', 'combined', 'popularity']],
    new_tv_sorted[['title', 'genres', 'overview', 'weighted_rate', 'combined', 'popularity']]
], ignore_index=True)

In [24]:
# Финальная очистка дубликатов
combined_df = combined_df.drop_duplicates(subset=['title'], keep='first')

In [25]:
# Добавление года для анализа
combined_df['year'] = combined_df['combined'].apply(extract_year_from_combined)

print(f"\nФинальный датасет: {len(combined_df)} записей")
print(f"\nРаспределение по годам:")


Финальный датасет: 5727 записей

Распределение по годам:


In [26]:
year_distribution = combined_df['year'].value_counts().sort_index()
print(year_distribution)

year
2020    1254
2021    1223
2022    1087
2023     447
2024    1570
2025     146
Name: count, dtype: int64


In [27]:
# Процент новых фильмов
new_content_count = len(combined_df[combined_df['year'].isin([2024, 2025])])
print(f"\nДоля контента 2024-2025: {new_content_count}/{len(combined_df)} ({new_content_count/len(combined_df)*100:.1f}%)")


Доля контента 2024-2025: 1716/5727 (30.0%)


In [28]:
# Сохранение
combined_df.to_csv('updated_combined_movies_tv_dataset_2024_2025.csv', index=False)
print("\nДатасет сохранен: 'updated_combined_movies_tv_dataset_2024_2025.csv'")


Датасет сохранен: 'updated_combined_movies_tv_dataset_2024_2025.csv'


In [29]:
# Дополнительный анализ новых релизов
print("\n=== АНАЛИЗ НОВЫХ РЕЛИЗОВ ===")
for year in [2024, 2025]:
    year_data = combined_df[combined_df['year'] == year]
    print(f"\n{year} год: {len(year_data)} записей")
    if len(year_data) > 0:
        print(f"Топ-10 по популярности:")
        top_10 = year_data.nlargest(10, 'popularity')[['title', 'weighted_rate', 'popularity']]
        print(top_10.to_string(index=False))
        
        # Сохранение списка
        year_data[['title', 'genres', 'weighted_rate']].to_csv(f'movies_tv_{year}.csv', index=False)


=== АНАЛИЗ НОВЫХ РЕЛИЗОВ ===

2024 год: 1570 записей
Топ-10 по популярности:
                        title  weighted_rate  popularity
      El amor no tiene receta       6.367568    2287.324
                  The Porters       5.149254    2197.590
       Ramez Gab Min El Akher       5.289855    2060.789
           Steelyard banknote       6.500000    2035.133
             Garye El Wohoush       6.500000    2017.947
                Divine Secret       6.500000    1764.021
          Khoyout Al Ma'azeeb       6.500000    1067.570
                         2024       6.500000     841.894
                   Strawberry       6.500000     822.910
Testament: The Story of Moses       6.201725     549.609

2025 год: 146 записей
Топ-10 по популярности:
                            title  weighted_rate  popularity
                            Moana            6.5      39.202
                   Fast X: Part 2            6.5      23.836
                         Avatar 3            6.5      14.635
    

In [30]:
# Проверка конкретных фильмов 2025
print("\n=== ПРОВЕРКА ФИЛЬМОВ 2025 ===")
test_titles = ["Superman", "Fantastic Four", "Captain America", "Avatar"]
for title in test_titles:
    found = combined_df[
        (combined_df['title'].str.contains(title, case=False, na=False)) & 
        (combined_df['year'] == 2025)
    ]
    if not found.empty:
        print(f"{title}: найден")
    else:
        # Проверяем в исходных данных
        in_original = movies_df[
            (movies_df['title'].str.contains(title, case=False, na=False)) & 
            (movies_df['year'] == 2025)
        ]
        if not in_original.empty:
            print(f"{title}: есть в исходных данных, но отфильтрован")
        else:
            print(f"{title}: отсутствует в исходных данных")


=== ПРОВЕРКА ФИЛЬМОВ 2025 ===
Superman: найден
Fantastic Four: найден
Captain America: есть в исходных данных, но отфильтрован
Avatar: найден


In [31]:
def find_movie_by_title(dataframe, title_query):
    """
    Ищет фильм/сериал в датафрейме по названию.
    
    Args:
        dataframe (pd.DataFrame): Датафрейм для поиска (например, combined_df).
        title_query (str): Название фильма/сериала для поиска.
                           Может быть полным названием или его частью.
                           
    Returns:
        pd.DataFrame: Датафрейм с найденными записями.
    """
    # Поиск по частичному совпадению, нечувствительно к регистру
    mask = dataframe['title'].str.contains(title_query, case=False, na=False)
    results = dataframe[mask]
    return results

In [34]:
found_movies = find_movie_by_title(combined_df, "Fantastic Four")

if not found_movies.empty:
    print(f"Найдено записей: {len(found_movies)}")
    # Выводим все колонки для найденных записей
    # Используем print(results.to_string()) для лучшего форматирования в консоли
    print(found_movies.to_string()) 
else:
    print("Фильм/сериал не найден в датасете.")

print("\n" + "="*50 + "\n")

Найдено записей: 1
               title             genres                                                                                                overview  weighted_rate                                                                                                                                                                                           combined  popularity  year
4079  Fantastic Four  [Science Fiction]  Set in the Marvel Cinematic Universe (MCU) and based on the Marvel Comics characters of the same name.            6.5  Title: Fantastic Four (Movie, 2025-04-30). Overview: Set in the Marvel Cinematic Universe (MCU) and based on the Marvel Comics characters of the same name. Genres: Science Fiction. Rating: 6.50       7.793  2025


